> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 08 · A chat portal for students (Flask)

**CampusMate v5 (agent with a front door)**  ·  ⏱️ Time budget: **10 min theory + 35 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Students won't call `agentcore invoke`. They need a web page. In this lab you run a small **Flask** web app that sends each message to your agent. The browser never sees AWS credentials; the Flask server calls AWS for it, like the fee counter clerk who talks to the accounts office on your behalf.

### What you'll do
- See how a browser session maps to a Runtime session and a memory actor
- Run the portal and chat in the browser
- See a clear error when the agent is unreachable
- Chat with a classmate's agent

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## How the pieces connect

```
Browser --(message + chosen student)--> Flask app.py --(invoke_agent_runtime)--> your agent on AgentCore Runtime
```
- **No login.** You pick a fictional student from a dropdown. That student ID becomes the **memory actor**, so each student is remembered separately.
- Each browser gets its own **session ID** per student (49 characters; Runtime needs at least 33).
- `projects/web/app.py` finds your agent by its name, `campusmate<your id>_default_agent`.

## 🔬 Live check: start the portal and send it a message, like a browser would

In [ ]:
import os, requests
PORT = 5051
server = subprocess.Popen([sys.executable, str(ROOT / "projects/web/app.py")],
                          env={**os.environ, "PORT": str(PORT), "CAMPUSMATE_AGENT_ARN": AGENT_ARN},   # which agent to call
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(40):
    try:
        print("health:", requests.get(f"http://127.0.0.1:{PORT}/api/health", timeout=1).json()); break
    except Exception:
        time.sleep(0.5)
browser = requests.Session()     # keeps cookies, like a real browser tab
data = browser.post(f"http://127.0.0.1:{PORT}/api/chat", json={"message": "Can I get my hall ticket?", "student_id": "AIDD-DEMO-005"},
                    timeout=180).json()
print("session:", data.get("session_id"), f"({len(data.get('session_id', ''))} characters)")
print("tools  :", [t["name"] for t in data.get("tools_used", [])])
print("answer :", data.get("result") or data.get("error"))

### Now use the real browser

1. Open `projects/web/app.py` and put your ID in the line `STUDENT_ID = "student01"` near the top.
2. In a terminal (venv active, repo root): `python projects/web/app.py`
3. Open **http://localhost:5050**. Pick **AIDD-DEMO-003** and ask *"Am I eligible for my exams?"*, then switch to **AIDD-DEMO-004**.
   Click **New session** and ask *"What did I ask before?"* (short-term memory resets; long-term memory stays).

## What an unreachable agent looks like

A good app explains the problem instead of crashing.

In [ ]:
bad = subprocess.Popen([sys.executable, str(ROOT / "projects/web/app.py")],
                       env={**os.environ, "PORT": str(PORT + 1),
                            "CAMPUSMATE_AGENT_ARN": AGENT_ARN.rsplit("/", 1)[0] + "/does_not_exist-0000000000"},
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    print(requests.post(f"http://127.0.0.1:{PORT + 1}/api/chat", json={"message": "hi"}, timeout=60).json())
finally:
    bad.terminate()

## ✍️ Your turn

**Chat with a classmate's agent.** Ask your neighbour for their agent ARN (the `agent` line printed near the top of their notebook) and start the portal pointing at it.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
CLASSMATE_ARN = "TODO"   # TODO: paste your neighbour's agent ARN

if CLASSMATE_ARN != "TODO":
    friend = subprocess.Popen([sys.executable, str(ROOT / "projects/web/app.py")],
                              env={**os.environ, "PORT": "5060", "CAMPUSMATE_AGENT_ARN": CLASSMATE_ARN},
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    print(requests.post("http://127.0.0.1:5060/api/chat", json={"message": "Hello! Which courses are in semester 3?",
                                                                 "student_id": "AIDD-DEMO-002"}, timeout=180).json())
    friend.terminate()
else:
    print("Skipped: set CLASSMATE_ARN first.")

In [ ]:
server.terminate()   # stop the portal from the live check

## ✅ Checkpoint

You are on track if:
- The live check printed a 49-character session ID and an answer about fees blocking the hall ticket
- The browser portal works at http://localhost:5050
- The bad ARN gave a friendly `Agent unreachable` message

**Stuck or behind?** Copy from `projects/web/` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `Address already in use` | Another app uses the port (AirPlay uses 5000 on Macs) | We use 5050; or `PORT=8000 python projects/web/app.py` |
| `No agent called campusmate_student01... found` | `STUDENT_ID` in `projects/web/app.py` not changed | Put your ID in app.py and restart it |
| Replies are slow the first time | A new session starts a new microVM (5-10 s) | Wait; the next message is faster |